<a href="https://colab.research.google.com/github/Khushi281300/token_retrieval/blob/main/notebooks/01_colab_reproduce_and_eval.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Topic 1: Uncertainty-Triggered RAG Across Decoders
**Author:** Khushi Shukla  


This notebook runs GPU evaluation for:
1. **Step 2:** Hashimoto et al. reproduction on TriviaQA (MSP, MTE, PRR)
2. **Step 4:** Multi-decoder dynamic RAG comparison on `allenai/OLMo-2-1124-7B-Instruct`

In [19]:
# 1. Environment & GPU Check
!nvidia-smi
!pip install -q transformers datasets accelerate scikit-learn scipy


Fri Nov 13 17:44:08 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   48C    P8             13W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [20]:
%cd /content/token_retrieval
!git pull origin main

/content/token_retrieval
From https://github.com/Khushi281300/token_retrieval
 * branch            main       -> FETCH_HEAD
Already up to date.


In [21]:
%cd /content/token_retrieval
!rm -rf /content/token_retrieval/token_retrieval
!git pull origin main

/content/token_retrieval
From https://github.com/Khushi281300/token_retrieval
 * branch            main       -> FETCH_HEAD
Already up to date.


In [22]:
# 3. Run Day-1 Smoke Test with OLMo-2-7B
!python smoke_test.py --model allenai/OLMo-2-1124-7B-Instruct --device cuda

=== Starting Smoke Test ===
Transformers version: 5.22.1
PyTorch version:      2.14.1+cu130
Operating System:     Linux-6.6.159+-x86_64-with-glibc2.39
Target Device:        cuda
Loading Model:        allenai/OLMo-2-1124-7B-Instruct ...
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100% 355/355 [00:59<00:00,  5.98it/s]
Model loaded successfully.
Prompt: 'The capital of France is'
Generated text: ' Paris. It is the largest city in France and one of the most populated cities in Europe. Paris'

[SUCCESS] Smoke test completed!
Results written to: runs/smoke_test.json
First 3 tokens preview:
  Step 0: token=' Paris' | raw_p=0.9185 | raw_H=0.6104
  Step 1: token='.' | raw_p=0.2971 | raw_H=1.9053
  Step 2: token=' It' | raw_p=0.2437 | raw_H=3.0554
Trigger check (raw max-entropy > 1.5): True


In [23]:
# 4. Step 2: Reproduce Hashimoto et al. on TriviaQA (Greedy Baseline)
from google.colab import userdata
userdata.get('HF_TOKEN')
# !python scripts/reproduce_hashimoto.py --model allenai/OLMo-2-1124-7B-Instruct --samples 50
!python scripts/reproduce_hashimoto.py --model allenai/OLMo-2-1124-7B-Instruct --samples 100 --unit nats

Hashimoto et al. Greedy Uncertainty Baseline (TriviaQA)
Model:        allenai/OLMo-2-1124-7B-Instruct
Samples:      100
Entropy Unit: nats
Loading weights: 100% 355/355 [00:51<00:00,  6.89it/s]
Some parameters are on the meta device because they were offloaded to the cpu.
Loading TriviaQA (rc.nocontext validation split, 100 items)...
Resolving data files: 100% 26/26 [00:00<00:00, 20105.44it/s]
Starting inference on 100 questions...

HASHIMOTO BASELINE EVALUATION SUMMARY
Total Questions Evaluated:       100
Mean Rouge-L:                    0.4901
Exact Match (EM):                42.00%
PRR x100 (MTE with Rouge-L):     13.96
PRR x100 (MSP with Rouge-L):     13.20
Full results saved to: runs/hashimoto_evaluation.json



In [ ]:
# 5. Phase 1 DEV ONLY: Calibrate thresholds on 500 dev questions, freeze tau, DON'T touch test
# Run this first. Check tau CIs. Then run step 6 (test) only after checking PROTOCOL.md margins.
import os
os.environ['HF_TOKEN'] = os.environ.get('HF_TOKEN', '') or __import__('google.colab', fromlist=['userdata']).userdata.get('HF_TOKEN', '')
!python scripts/run_experiments.py \
    --model allenai/OLMo-2-1124-7B-Instruct \
    --dataset trivia_qa \
    --num_dev 500 \
    --num_test 1000 \
    --max_draft_tokens 64 \
    --max_answer_tokens 32 \
    --score_type raw \
    --load_in_4bit \
    --run_dev_only

In [ ]:
# 6. FULL RUN (test set) -- ONLY run after dev calibration is done and margins are set in PROTOCOL.md
# DO NOT run this cell until Phase 3 (dev) is complete and equivalence margins are frozen.
!python scripts/run_experiments.py \
    --model allenai/OLMo-2-1124-7B-Instruct \
    --dataset trivia_qa \
    --num_dev 500 \
    --num_test 1000 \
    --max_draft_tokens 64 \
    --max_answer_tokens 32 \
    --score_type raw \
    --load_in_4bit